In [ ]:
from pathlib import Path
import pickle

import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()


def find_release_root(start: Path) -> Path:
    sentinel_dirs = ('Step0_Data', 'Step9_FutureProjection')
    for candidate in [start, *start.parents]:
        if all((candidate / name).is_dir() for name in sentinel_dirs):
            return candidate
        nested_project = candidate / 'Global-solid-waste-prediction'
        if all((nested_project / name).is_dir() for name in sentinel_dirs):
            return nested_project
    raise FileNotFoundError('Could not locate Global-solid-waste-prediction repository root from the current working directory')


release_root = find_release_root(cwd)
output_dir = release_root / 'Step9_FutureProjection' / '2_Results'
step6_dir = release_root / 'Step7_ModelTraining' / '2_Results'
historical_path = release_root / 'Step8_HistoricalCompletion' / '2_Results' / '0_historical_panel_completed.csv'
driver_panel_path = output_dir / '0_future_driver_panel.csv'
subset_path = output_dir / '0_future_country_subset_included.csv'
BATCH_SIZE = 512
PREDICT_PROGRESS_EVERY = 2
WASTE_TYPES = ['AW', 'CDW', 'IW', 'MSW']


In [ ]:
def predict_in_batches(model, features, batch_size=4096, progress_every=4):
    n_rows = len(features)
    if n_rows == 0:
        return np.asarray([], dtype=float)
    if batch_size is None or batch_size <= 0 or batch_size >= n_rows:
        return np.asarray(model.predict(features), dtype=float).reshape(-1)
    predictions = []
    n_batches = (n_rows - 1) // batch_size + 1
    for batch_idx in range(n_batches):
        start = batch_idx * batch_size
        end = min((batch_idx + 1) * batch_size, n_rows)
        batch_features = features.iloc[start:end] if hasattr(features, 'iloc') else features[start:end]
        batch_pred = np.asarray(model.predict(batch_features), dtype=float).reshape(-1)
        predictions.append(batch_pred)
        if progress_every and (((batch_idx + 1) % progress_every == 0) or (batch_idx + 1 == n_batches)):
            print(f'Prediction batch {batch_idx + 1}/{n_batches} ({end}/{n_rows} rows)')
    return np.concatenate(predictions, axis=0)

def clone_hist_across_scenarios(hist_subset, scenarios):
    hist_rows = []
    for scenario in scenarios:
        part = hist_subset.copy()
        part['Scenario'] = scenario
        hist_rows.append(part)
    return pd.concat(hist_rows, ignore_index=True)

def assert_unique_keys(df, keys, label):
    duplicated = df.duplicated(keys, keep=False)
    if duplicated.any():
        sample = df.loc[duplicated, keys].head(10).to_dict('records')
        raise RuntimeError(f'{label} contains non-unique keys for {keys}: {sample}')

def validate_prediction_keys(predictions_df):
    key_cols = ['Country Code', 'Year', 'Scenario', 'WasteFlag']
    assert_unique_keys(predictions_df, key_cols, 'predictions_df')
    unknown_flags = sorted(set(predictions_df['WasteFlag'].dropna().astype(str)) - set(WASTE_TYPES))
    if unknown_flags:
        raise RuntimeError(f'predictions_df contains unexpected WasteFlag values: {unknown_flags}')
    coverage = predictions_df.groupby(['Country Code', 'Year', 'Scenario'])['WasteFlag'].agg(lambda s: sorted(set(s.astype(str))))
    invalid = coverage[coverage.apply(lambda flags: flags != sorted(WASTE_TYPES))]
    if not invalid.empty:
        sample = invalid.head(10).to_dict()
        raise RuntimeError(f'predictions_df does not cover exactly 4 waste types per future key: {sample}')

def parse_included_mask(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    numeric = pd.Series(pd.to_numeric(series, errors='coerce'), index=series.index)
    mask = pd.Series(False, index=series.index)
    mask = mask | numeric.fillna(0).astype(float).ne(0.0)
    normalized = series.astype(str).str.strip().str.lower()
    mask = mask | normalized.isin({'true', '1', 'yes', 'y', 'included'})
    return mask

def assert_matching_key_sets(left_df, right_df, keys, left_label, right_label):
    left_keys = left_df[keys].drop_duplicates().copy()
    right_keys = right_df[keys].drop_duplicates().copy()
    missing_in_right = left_keys.merge(right_keys, on=keys, how='left', indicator=True)
    missing_in_right = missing_in_right[missing_in_right['_merge'] == 'left_only'][keys]
    if not missing_in_right.empty:
        sample = missing_in_right.head(10).to_dict('records')
        raise RuntimeError(f'{left_label} has {len(missing_in_right)} keys missing in {right_label}: {sample}')
    extra_in_right = right_keys.merge(left_keys, on=keys, how='left', indicator=True)
    extra_in_right = extra_in_right[extra_in_right['_merge'] == 'left_only'][keys]
    if not extra_in_right.empty:
        sample = extra_in_right.head(10).to_dict('records')
        raise RuntimeError(f'{right_label} has {len(extra_in_right)} extra keys not found in {left_label}: {sample}')

def build_future_panel_from_predictions(driver_df, predictions_df, source_label):
    key_cols = ['Country Code', 'Year', 'Scenario']
    driver_cols = ['Country Code', 'Year', 'Scenario', 'IncomeGroup', 'Region', 'SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST']
    future_panel = driver_df[driver_cols].copy()
    assert_unique_keys(future_panel, key_cols, 'driver_df')
    validate_prediction_keys(predictions_df)
    pred_wide = predictions_df.pivot(index=key_cols, columns='WasteFlag', values='Predicted_Raw').reset_index()
    pred_wide.columns.name = None
    missing_waste_cols = [wt for wt in WASTE_TYPES if wt not in pred_wide.columns]
    if missing_waste_cols:
        raise RuntimeError(f'predictions_df pivot is missing waste columns: {missing_waste_cols}')
    pred_wide = pred_wide.rename(columns={wt: f'{wt}_pred' for wt in WASTE_TYPES})
    assert_unique_keys(pred_wide, key_cols, 'predictions_wide_df')
    assert_matching_key_sets(future_panel, pred_wide, key_cols, 'driver_df', 'predictions_wide_df')
    future_panel = future_panel.merge(pred_wide, on=key_cols, how='left')
    pred_na_counts = {f'{wt}_pred': int(future_panel[f'{wt}_pred'].isna().sum()) for wt in WASTE_TYPES}
    pred_na_counts = {col: count for col, count in pred_na_counts.items() if count > 0}
    if pred_na_counts:
        raise RuntimeError(f'future_panel contains NaN predictions after merge: {pred_na_counts}')
    for wt in WASTE_TYPES:
        future_panel[f'{wt}_final'] = future_panel[f'{wt}_pred']
        future_panel[f'{wt}_source'] = source_label
        future_panel[f'{wt}_t'] = np.nan
    return future_panel

def assemble_super_panel(hist_super, future_panel):
    super_cols = ['Country Code', 'Year', 'Scenario', 'IncomeGroup', 'Region', 'SP.POP.TOTL', 'NY.GDP.MKTP.PP.KD', 'NY.GDP.PCAP.PP.KD', 'EN.POP.DNST']
    for wt in WASTE_TYPES:
        super_cols.extend([f'{wt}_t', f'{wt}_pred', f'{wt}_final', f'{wt}_source'])
    normalized_parts = []
    for label, frame in [('hist_super', hist_super), ('future_panel', future_panel)]:
        missing_cols = [col for col in super_cols if col not in frame.columns]
        if missing_cols:
            for col in missing_cols:
                frame[col] = np.nan
        normalized = frame[super_cols].copy()
        normalized['Scenario'] = normalized['Scenario'].astype(str)
        normalized_year = pd.Series(pd.to_numeric(normalized['Year'], errors='raise'), index=normalized.index)
        normalized['Year'] = normalized_year.astype(int)
        normalized_parts.append(normalized)
    return pd.concat(normalized_parts, ignore_index=True).sort_values(['Scenario', 'Country Code', 'Year']).reset_index(drop=True)


In [ ]:
pred_parts = []
feature_cols = pd.read_csv(step6_dir / '0_feature_contract_processed.csv')['feature'].astype(str).tolist()
for waste_type in WASTE_TYPES:
    x = pd.read_csv(output_dir / f'0_future_features_transformed_{waste_type}.csv')
    with open(step6_dir / f'1_finalize_model_{waste_type}.pkl', 'rb') as handle:
        model = pickle.load(handle)
    preds_log = predict_in_batches(model, x[feature_cols].copy(), batch_size=BATCH_SIZE, progress_every=PREDICT_PROGRESS_EVERY)
    part = x[['Country Code', 'Year', 'Scenario', 'WasteFlag']].copy()
    part['Predicted_Raw'] = np.expm1(preds_log)
    part['Predicted_Log'] = preds_log
    pred_parts.append(part)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
pred = pd.concat(pred_parts, ignore_index=True).sort_values(['Country Code', 'Year', 'Scenario', 'WasteFlag']).reset_index(drop=True)
validate_prediction_keys(pred)
pred.to_csv(output_dir / '1_ssp_predictions_long.csv', index=False)
pred.head()


In [ ]:
hist = pd.read_csv(historical_path)
driver = pd.read_csv(driver_panel_path)
subset = pd.read_csv(subset_path)
included_mask = parse_included_mask(subset['included'])
included_codes = subset.loc[included_mask, ['Country Code']].copy()
included_codes['Country Code'] = included_codes['Country Code'].astype(str)
included_iso3 = included_codes['Country Code'].dropna().tolist()
hist.to_csv(output_dir / '1_dataset_a_historical_full_182.csv', index=False)
hist_subset = hist[hist['Country Code'].isin(included_iso3)].copy()
scenarios = sorted(pred['Scenario'].dropna().astype(str).unique().tolist())
hist_super = clone_hist_across_scenarios(hist_subset, scenarios)
future_panel = build_future_panel_from_predictions(driver[driver['Year'] >= 2025].copy(), pred[pred['Year'] >= 2025].copy(), 'ssp_projected')
super_panel = assemble_super_panel(hist_super, future_panel)
super_panel.to_csv(output_dir / '1_super_panel_1990_2100.csv', index=False)
super_panel.shape
